# Test sample sizes for global mortality

This script calculates the global mortality for one year of one ensemble member for a range of bootstrap replications and creates several plots to evaluate them. To do this `2_Save_sample_size.ipynb` needs to be run for each of the sample ranges.

In [ ]:
import os
import xarray as xr
import warnings
import seaborn as sns
from utils.utils import get_scenario_config, autosize_figure
from utils.mortality_utils import att_frac
import matplotlib.pyplot as plt

In [ ]:
# === Path config ===
MASKS_DIR = "/glade/work/awells/workflow/BMR/masks/country/"
POP_DIR = "/glade/work/awells/workflow/SSP_pop/SSP2/"
TMREL_DIR = "/glade/derecho/scratch/awells/workflow/TMREL/"
BETA_DIR = "/glade/derecho/scratch/awells/workflow/beta_ozone/"
BMR_DIR = "/glade/derecho/scratch/awells/workflow/BMR_ozone/"

In [ ]:
# Load country masks
mask_file = "GBD_Country_Masks_0.10.nc"
mask_path = os.path.join(MASKS_DIR, mask_file)
masks = xr.open_dataarray(mask_path)

# Load population file
pop_file = "ssp2_coarse_grid_annual_2000-2100.nc"
pop_path = os.path.join(POP_DIR, pop_file)
population = xr.open_dataarray(pop_path)
pop = population.reindex_like(masks, method="nearest", tolerance=1e-9)

Before running this script, make sure `2__Save_sample_size.ipynb` has been run for the number of samples you are testing.

In [ ]:
# Select a range of sample sizes
sample_range = [200, 300, 500, 1000]
# Run for one year
year = 2035
# Run for one ensemble member
ens_num = 1

In [ ]:
warnings.filterwarnings('ignore')

# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "SSP245"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]

O3_DIR = f"/glade/work/awells/workflow/{model}/ozone/OSDMA8_BC/"

print(f"Processing ensemble member {ens_num:02d}")
# {years.stop - 1} from OSDMA8 calculation
dates = f"{years.start}-{years.stop - 1}"

# Load ozone data
o3_file = f"OSDMA8_BC_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
o3_path = os.path.join(O3_DIR, o3_file)
o3 = xr.open_dataarray(o3_path).astype("float32")
o3 = o3.reindex_like(masks, method="nearest", tolerance=1e-9, fill_value=0)

del o3_file, o3_path

# make o3 dask-backed
o3 = o3.chunk({'lat': 180, 'lon': 360})

for n_samples in sample_range:
    print(f"Processing {n_samples} samples")
    # TMREL from GBD21 (uniform distribution)
    tmrel_file = f"TMREL_{n_samples}_samples_ozone.nc"
    tmrel_path = os.path.join(TMREL_DIR, tmrel_file)
    tmrel_da = xr.open_dataarray(tmrel_path)

    # Beta from RR per 10ppb (normal distribution)
    beta_file = f"beta_{n_samples}_samples_ozone.nc"
    beta_path = os.path.join(BETA_DIR, beta_file)
    beta_da = xr.open_dataarray(beta_path)

    # Load BMR for each grid point (normal distribution)
    bmr_file = f"GBD_BMR_Country_Mask_COPD_{n_samples}_samples_1990-2009.nc"
    bmr_path = os.path.join(BMR_DIR, bmr_file)
    BMR = xr.open_dataarray(bmr_path)

    print(f"Processing year {year}")
    o3_year = o3.sel(year=year)
    # Calculation the attributable fraction
    AF = att_frac(o3_year, tmrel_da, beta_da).chunk({"samples": 10})

    del tmrel_da, beta_da

    POP = pop.sel(year=year).chunk({"lat": 180, "lon": 360})
    # Calculate mortality at each grid point for n samples
    M = AF * BMR * POP
    # Calculate the total global mortality
    global_M = M.sum(dim=("lat", "lon"))

    del o3_year, AF, POP, M, BMR

    description = ("Global mortality (COPD) due to ozone "
                   " - scripts by A.F. Wells (2025)")
    global_M.attrs["description"] = description
    global_M.attrs["model"] = model
    global_M.attrs["scenario"] = scenario
    global_M.attrs["ensemble_number"] = ens_num
    global_M.attrs["year"] = year

    SAVE_DIR = f"/glade/work/awells/workflow/{model}/mortality/ozone/global/{n_samples}_samples/"
    out_file = f"Global_mortality_{n_samples}samples_{model}_{scenario}_{ens_num:02d}_{year}.nc"
    out_path = os.path.join(SAVE_DIR, out_file)
    print(f"Saving to {out_path}")
    global_M.to_netcdf(out_path)

    del global_M

del o3

print("All processing complete.")

Plot figures to evaluate the number of bootstrap replications.

In [ ]:
plt.figure(figsize=autosize_figure(1, 1, xscale_factor=1.5))

das = []
for n_samples in sample_range[:-1]:
    DIR = f"/glade/work/awells/workflow/{model}/mortality/ozone/global/{n_samples}_samples/"
    file = f"Global_mortality_{n_samples}samples_{model}_{scenario}_{ens_num:02d}_{year}.nc"
    file_path = os.path.join(DIR, file)
    da = xr.open_dataarray(file_path)
    das.append(da)

for da in das:
    da.plot(label=len(da))

plt.legend()
plt.title(f"Global mortality samples, {model} {scenario} {year}")
plt.ylabel("Total Global Mortality")

plt.show()

In [ ]:
# Compare distributions for each sample size
fig, axs = plt.subplots(2, 3, figsize=autosize_figure(2, 3))

# 0 vs 1
sns.kdeplot(das[0].values, fill=True, label=f"{sample_range[0]} samples", ax=axs[0, 0])
sns.kdeplot(das[1].values, fill=True, label=f"{sample_range[1]} samples", ax=axs[0, 0])
axs[0, 0].axvline(das[0].median(), color="tab:blue", linestyle="-")
axs[0, 0].axvline(das[1].median(), color="tab:orange", linestyle="--")
axs[0, 0].axvline(das[0].quantile(0.025, 'samples'), color="tab:blue", linestyle="-")
axs[0, 0].axvline(das[1].quantile(0.025, 'samples'), color="tab:orange", linestyle="--")
axs[0, 0].axvline(das[0].quantile(0.975, 'samples'), color="tab:blue", linestyle="-")
axs[0, 0].axvline(das[1].quantile(0.975, 'samples'), color="tab:orange", linestyle="--")
axs[0, 0].legend()
axs[0, 0].set_title(f"{sample_range[0]} vs {sample_range[1]}")

# 0 vs 2
sns.kdeplot(das[0].values, fill=True, label=f"{sample_range[0]} samples", ax=axs[0, 1])
sns.kdeplot(das[2].values, fill=True, label=f"{sample_range[2]} samples", ax=axs[0, 1])
axs[0, 1].axvline(das[0].median(), color="tab:blue", linestyle="-")
axs[0, 1].axvline(das[2].median(), color="tab:orange", linestyle="--")
axs[0, 1].axvline(das[0].quantile(0.025, 'samples'), color="tab:blue", linestyle="-")
axs[0, 1].axvline(das[2].quantile(0.025, 'samples'), color="tab:orange", linestyle="--")
axs[0, 1].axvline(das[0].quantile(0.975, 'samples'), color="tab:blue", linestyle="-")
axs[0, 1].axvline(das[2].quantile(0.975, 'samples'), color="tab:orange", linestyle="--")
axs[0, 1].legend()
axs[0, 1].set_title(f"{sample_range[0]} vs {sample_range[2]}")

# 0 vs 3
sns.kdeplot(das[0].values, fill=True, label=f"{sample_range[0]} samples", ax=axs[0, 2])
sns.kdeplot(das[3].values, fill=True, label=f"{sample_range[3]} samples", ax=axs[0, 2])
axs[0, 2].axvline(das[0].median(), color="tab:blue", linestyle="-")
axs[0, 2].axvline(das[3].median(), color="tab:orange", linestyle="--")
axs[0, 2].axvline(das[0].quantile(0.025, 'samples'), color="tab:blue", linestyle="-")
axs[0, 2].axvline(das[3].quantile(0.025, 'samples'), color="tab:orange", linestyle="--")
axs[0, 2].axvline(das[0].quantile(0.975, 'samples'), color="tab:blue", linestyle="-")
axs[0, 2].axvline(das[3].quantile(0.975, 'samples'), color="tab:orange", linestyle="--")
axs[0, 2].legend()
axs[0, 2].set_title(f"{sample_range[0]} vs {sample_range[3]}")

# 1 vs 2
sns.kdeplot(das[1].values, fill=True, label=f"{sample_range[1]} samples", ax=axs[1, 0])
sns.kdeplot(das[2].values, fill=True, label=f"{sample_range[2]} samples", ax=axs[1, 0])
axs[1, 0].axvline(das[1].median(), color="tab:blue", linestyle="-")
axs[1, 0].axvline(das[2].median(), color="tab:orange", linestyle="--")
axs[1, 0].axvline(das[1].quantile(0.025, 'samples'), color="tab:blue", linestyle="-")
axs[1, 0].axvline(das[2].quantile(0.025, 'samples'), color="tab:orange", linestyle="--")
axs[1, 0].axvline(das[1].quantile(0.975, 'samples'), color="tab:blue", linestyle="-")
axs[1, 0].axvline(das[2].quantile(0.975, 'samples'), color="tab:orange", linestyle="--")
axs[1, 0].legend()
axs[1, 0].set_title(f"{sample_range[1]} vs {sample_range[2]}")

# 1 vs 3
sns.kdeplot(das[1].values, fill=True, label=f"{sample_range[1]} samples", ax=axs[1, 1])
sns.kdeplot(das[3].values, fill=True, label=f"{sample_range[3]} samples", ax=axs[1, 1])
axs[1, 1].axvline(das[1].median(), color="tab:blue", linestyle="-")
axs[1, 1].axvline(das[3].median(), color="tab:orange", linestyle="--")
axs[1, 1].axvline(das[1].quantile(0.025, 'samples'), color="tab:blue", linestyle="-")
axs[1, 1].axvline(das[3].quantile(0.025, 'samples'), color="tab:orange", linestyle="--")
axs[1, 1].axvline(das[1].quantile(0.975, 'samples'), color="tab:blue", linestyle="-")
axs[1, 1].axvline(das[3].quantile(0.975, 'samples'), color="tab:orange", linestyle="--")
axs[1, 1].legend()
axs[1, 1].set_title(f"{sample_range[1]} vs {sample_range[3]}")

# 2 vs 3
sns.kdeplot(das[2].values, fill=True, label=f"{sample_range[2]} samples", ax=axs[1, 2])
sns.kdeplot(das[3].values, fill=True, label=f"{sample_range[3]} samples", ax=axs[1, 2])
axs[1, 2].axvline(das[2].median(), color="tab:blue", linestyle="-")
axs[1, 2].axvline(das[3].median(), color="tab:orange", linestyle="--")
axs[1, 2].axvline(das[2].quantile(0.025, 'samples'), color="tab:blue", linestyle="-")
axs[1, 2].axvline(das[3].quantile(0.025, 'samples'), color="tab:orange", linestyle="--")
axs[1, 2].axvline(das[2].quantile(0.975, 'samples'), color="tab:blue", linestyle="-")
axs[1, 2].axvline(das[3].quantile(0.975, 'samples'), color="tab:orange", linestyle="--")
axs[1, 2].legend()
axs[1, 2].set_title(f"{sample_range[2]} vs {sample_range[3]}")

plt.tight_layout()
plt.show()